# Introduction to Causal Bayesian Networks

This notebook introduces the basic concepts of Causal Bayesian Networks and demonstrates how to build and query them for counterfactual analysis.

## Contents
1. Building a simple causal model
2. Performing interventions
3. Counterfactual reasoning
4. Decision analysis

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pgmpy.factors.discrete import TabularCPD

from src.models.bayesian_network import CausalBayesianNetwork
from src.models.structural_causal_model import StructuralCausalModel
from src.inference.counterfactual import CounterfactualInference
from src.inference.intervention import InterventionAnalysis

## Example 1: Medical Treatment Decision

Let's model a medical scenario:
- **Disease** (D): Whether patient has the disease
- **Treatment** (T): Whether treatment is given
- **Recovery** (R): Whether patient recovers

Causal structure: D → T → R, D → R

In [ ]:
# Create the network structure
cbn = CausalBayesianNetwork()
cbn.add_nodes(['Disease', 'Treatment', 'Recovery'])
cbn.add_edges([('Disease', 'Treatment'), ('Disease', 'Recovery'), ('Treatment', 'Recovery')])

# Define CPDs
cpd_d = TabularCPD('Disease', 2, [[0.7], [0.3]])

cpd_t = TabularCPD('Treatment', 2, 
                   [[0.9, 0.1],   # P(T=0|D=0), P(T=0|D=1)
                    [0.1, 0.9]],  # P(T=1|D=0), P(T=1|D=1)
                   evidence=['Disease'], evidence_card=[2])

cpd_r = TabularCPD('Recovery', 2,
                   [[0.95, 0.8, 0.5, 0.1],  # P(R=0|...)
                    [0.05, 0.2, 0.5, 0.9]], # P(R=1|...)
                   evidence=['Disease', 'Treatment'], 
                   evidence_card=[2, 2])

cbn.set_cpd('Disease', cpd_d)
cbn.set_cpd('Treatment', cpd_t)
cbn.set_cpd('Recovery', cpd_r)

# Build the model
cbn.build_model()
print(f"Network created: {cbn}")
print(f"Valid: {cbn.validate()}")

## Intervention Analysis: Effect of Treatment

Let's analyze: **What is the average treatment effect?**

In [ ]:
intervention = InterventionAnalysis(cbn)

# Compute average treatment effect
ate = intervention.average_treatment_effect('Treatment', 'Recovery')
print(f"Average Treatment Effect: {ate:.3f}")
print(f"\nInterpretation: Treatment increases recovery probability by {ate*100:.1f} percentage points")

# Compare interventional distributions
p_r_treat = intervention.do_intervention({'Treatment': 1}, 'Recovery')
p_r_no_treat = intervention.do_intervention({'Treatment': 0}, 'Recovery')

print(f"\nP(Recovery | do(Treatment=1)): {p_r_treat}")
print(f"P(Recovery | do(Treatment=0)): {p_r_no_treat}")

## Counterfactual Reasoning

**Counterfactual question**: A patient with disease did not receive treatment and did not recover. Would they have recovered if they had received treatment?

This requires a Structural Causal Model.

In [ ]:
# Create SCM with structural equations
scm = StructuralCausalModel(cbn)

# Define structural equations (simplified linear model)
scm.add_structural_equation(
    'Disease',
    lambda parents, noise: (noise > 0).astype(int)
)

scm.add_structural_equation(
    'Treatment',
    lambda parents, noise: (parents.get('Disease', 0) + noise > 0.5).astype(int)
)

scm.add_structural_equation(
    'Recovery',
    lambda parents, noise: (
        0.9 * parents.get('Treatment', 0) - 
        0.6 * parents.get('Disease', 0) + noise > 0
    ).astype(int)
)

# Perform counterfactual inference
cf_inference = CounterfactualInference(cbn, scm)

observed = {'Disease': 1, 'Treatment': 0, 'Recovery': 0}
intervention_cf = {'Treatment': 1}

result = cf_inference.query_counterfactual(
    observed=observed,
    intervention=intervention_cf,
    target='Recovery',
    method='scm'
)

print(f"Observed: {observed}")
print(f"Counterfactual (if Treatment=1): Recovery = {result}")
print(f"\nConclusion: {'Treatment would have helped' if result == 1 else 'Treatment would not have helped'}")

## Probability of Necessity and Sufficiency

- **PN**: Probability treatment was necessary for recovery
- **PS**: Probability treatment is sufficient for recovery

In [1]:
# Note: These require Monte Carlo simulation for accurate estimates
n_samples = 1000
pn_samples = []
ps_samples = []

for _ in range(n_samples):
    try:
        pn = cf_inference.probability_of_necessity('Treatment', 'Recovery')
        pn_samples.append(pn)
    except:
        pass
    
    try:
        ps = cf_inference.probability_of_sufficiency('Treatment', 'Recovery')
        ps_samples.append(ps)
    except:
        pass

if pn_samples:
    print(f"Probability of Necessity: {np.mean(pn_samples):.3f}")
if ps_samples:
    print(f"Probability of Sufficiency: {np.mean(ps_samples):.3f}")

## Visualization

Visualize the causal structure

In [ ]:
import networkx as nx

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(cbn.dag, seed=42)
nx.draw(cbn.dag, pos, 
        with_labels=True, 
        node_color='lightblue',
        node_size=3000,
        font_size=12,
        font_weight='bold',
        arrows=True,
        arrowsize=20,
        edge_color='gray')
plt.title('Causal Graph: Medical Treatment Example', fontsize=14)
plt.tight_layout()
plt.show()

## Exercises

1. Modify the CPDs and observe how treatment effects change
2. Add a confounding variable (e.g., Age) and analyze its impact
3. Compute conditional treatment effects for different subpopulations
4. Design a decision policy that maximizes expected recovery